## GECToR-like Error generation

In this notebook, we explore the possibilities of GECToR pipeline for data augmentation. We create our own dictionaries that we later use to corrupt the sentences using the GECToR code (error.py)

In [56]:
!git clone https://github.com/awasthiabhijeet/PIE.git

Cloning into 'PIE'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 132 (delta 30), reused 20 (delta 20), pack-reused 92
Receiving objects: 100% (132/132), 2.44 MiB | 11.61 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [57]:
%cd /content/PIE

/content/PIE


In [58]:
%cd errorify

/content/PIE/errorify


In [82]:
import math
import pickle
import random
from numpy.random import choice as npchoice
import pandas as pd
from collections import defaultdict

In [59]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="FacebookAI/xlm-roberta-base")

Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
sents_file = pd.read_csv('/content/span_sentences.tsv', sep='\t')
sents_file = sents_file.rename(columns={'Copying expression':'Tag',
                                        'middle place':'FOE',
                                        'Agriculture in Indonesia is in the middle place and the last one is under "industry" control.':'Sentence'})
sents_file = sents_file.dropna()

In [46]:
sents_file['masked_sents'] = sents_file.apply(lambda row: row['Sentence'].replace(str(row['FOE']), '<mask>', 1), axis=1)

# All Errors

In [47]:
sentence_list = list(sents_file['masked_sents'])

In [48]:
len(sentence_list)

2212

In [54]:
corrections_list = []
for sent_mask in sentence_list:
    try:
        corrections_list.append(pipe(sent_mask)[0]['token_str'])
    except Exception:
        corrections_list.append('XXX')

In [49]:
error_list = list(sents_file['FOE'])

In [50]:
len(error_list)

2212

In [55]:
len(corrections_list)

2212

In [65]:
len(set(error_list))

868

In [70]:
len(set(corrections_list))

715

In [80]:
L1_REPLACES = {}
for i in range(len(corrections_list)):
    error = error_list[i]
    correction = corrections_list[i]
    if correction in L1_REPLACES:
        L1_REPLACES[correction][error] += 1
    if correction not in L1_REPLACES:
        L1_REPLACES[correction] = defaultdict(int)
        L1_REPLACES[correction][error] += 1

In [ ]:
L1_REPLACES

In [83]:
with open('L1_replaces_all.p', 'wb') as handle:
    pickle.dump(L1_REPLACES, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Word form transmission

In [84]:
sentence_list_WFT = list(sents_file[sents_file['Tag'] == 'Word form transmission']['masked_sents'])

In [91]:
len(sentence_list_WFT)

239

In [101]:
error_list_WFT = list(sents_file[sents_file['Tag'] == 'Word form transmission']['FOE'])

In [102]:
len(error_list_WFT)

239

In [103]:
corrections_list_WFT = []
for sent_mask in sentence_list_WFT:
    try:
        corrections_list_WFT.append(pipe(sent_mask)[0]['token_str'])
    except Exception:
        corrections_list_WFT.append('XXX')

In [128]:
L1_REPLACES_WFT = {}
for i in range(len(corrections_list_WFT)):
    error = error_list_WFT[i]
    correction = corrections_list_WFT[i]
    if correction in L1_REPLACES_WFT:
        L1_REPLACES_WFT[correction][error] += 1
    if correction not in L1_REPLACES_WFT:
        L1_REPLACES_WFT[correction] = defaultdict(int)
        L1_REPLACES_WFT[correction][error] += 1

In [129]:
with open('L1_replaces_WFT.p', 'wb') as handle:
    pickle.dump(L1_REPLACES_WFT, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Synonyms

In [86]:
sentence_list_Syn = list(sents_file[sents_file['Tag'] == 'Synonyms']['masked_sents'])

In [92]:
len(sentence_list_Syn)

903

In [107]:
error_list_Syn = list(sents_file[sents_file['Tag'] == 'Synonyms']['FOE'])

In [108]:
len(error_list_Syn)

903

In [109]:
corrections_list_Syn = []
for sent_mask in sentence_list_Syn:
    try:
        corrections_list_Syn.append(pipe(sent_mask)[0]['token_str'])
    except Exception:
        corrections_list_Syn.append('XXX')

In [130]:
L1_REPLACES_SYN = {}
for i in range(len(corrections_list_Syn)):
    error = error_list_Syn[i]
    correction = corrections_list_Syn[i]
    if correction in L1_REPLACES_SYN:
        L1_REPLACES_SYN[correction][error] += 1
    if correction not in L1_REPLACES_SYN:
        L1_REPLACES_SYN[correction] = defaultdict(int)
        L1_REPLACES_SYN[correction][error] += 1

In [131]:
with open('L1_replaces_syn.p', 'wb') as handle:
    pickle.dump(L1_REPLACES_SYN, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Copying expression

In [95]:
sentence_list_CopExp = list(sents_file[sents_file['Tag'] == 'Copying expression']['masked_sents'])

In [112]:
len(sentence_list_CopExp)

383

In [113]:
error_list_CopExp = list(sents_file[sents_file['Tag'] == 'Copying expression']['FOE'])

In [114]:
len(error_list_CopExp)

383

In [115]:
corrections_list_CopExp = []
for sent_mask in sentence_list_CopExp:
    try:
        corrections_list_CopExp.append(pipe(sent_mask)[0]['token_str'])
    except Exception:
        corrections_list_CopExp.append('XXX')

In [134]:
L1_REPLACES_COPEXP = {}
for i in range(len(corrections_list_CopExp)):
    error = error_list_CopExp[i]
    correction = corrections_list_CopExp[i]
    if correction in L1_REPLACES_COPEXP:
        L1_REPLACES_COPEXP[correction][error] += 1
    if correction not in L1_REPLACES_COPEXP:
        L1_REPLACES_COPEXP[correction] = defaultdict(int)
        L1_REPLACES_COPEXP[correction][error] += 1

In [135]:
with open('L1_replaces_copexp.p', 'wb') as handle:
    pickle.dump(L1_REPLACES_COPEXP, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Tense semantics

In [89]:
sentence_list_TenSem = list(sents_file[sents_file['Tag'] == 'Tense semantics']['masked_sents'])

In [97]:
len(sentence_list_TenSem)

543

In [118]:
error_list_TenSem = list(sents_file[sents_file['Tag'] == 'Tense semantics']['FOE'])

In [119]:
len(error_list_TenSem)

543

In [120]:
corrections_list_TenSem = []
for sent_mask in sentence_list_TenSem:
    try:
        corrections_list_TenSem.append(pipe(sent_mask)[0]['token_str'])
    except Exception:
        corrections_list_TenSem.append('XXX')

In [136]:
L1_REPLACES_TENSEM = {}
for i in range(len(corrections_list_TenSem)):
    error = error_list_TenSem[i]
    correction = corrections_list_TenSem[i]
    if correction in L1_REPLACES_TENSEM:
        L1_REPLACES_TENSEM[correction][error] += 1
    if correction not in L1_REPLACES_TENSEM:
        L1_REPLACES_TENSEM[correction] = defaultdict(int)
        L1_REPLACES_TENSEM[correction][error] += 1

In [137]:
with open('L1_replaces_tensem.p', 'wb') as handle:
    pickle.dump(L1_REPLACES_TENSEM, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Transliteration

In [99]:
sentence_list_transliteration = list(sents_file[sents_file['Tag'] == 'Transliteration']['masked_sents'])

In [100]:
len(sentence_list_transliteration)

134

In [123]:
error_list_transliteration = list(sents_file[sents_file['Tag'] == 'Transliteration']['FOE'])

In [124]:
len(error_list_transliteration)

134

In [125]:
corrections_list_transliteration = []
for sent_mask in sentence_list_transliteration:
    try:
        corrections_list_transliteration.append(pipe(sent_mask)[0]['token_str'])
    except Exception:
        corrections_list_transliteration.append('XXX')

In [138]:
L1_REPLACES_TRANSL = {}
for i in range(len(corrections_list_transliteration)):
    error = error_list_transliteration[i]
    correction = corrections_list_transliteration[i]
    if correction in L1_REPLACES_TRANSL:
        L1_REPLACES_TRANSL[correction][error] += 1
    if correction not in L1_REPLACES_TRANSL:
        L1_REPLACES_TRANSL[correction] = defaultdict(int)
        L1_REPLACES_TRANSL[correction][error] += 1

In [139]:
with open('L1_replaces_transliteration.p', 'wb') as handle:
    pickle.dump(L1_REPLACES_TRANSL, handle, protocol=pickle.HIGHEST_PROTOCOL)

# End Result Class Creation

In [140]:
REPLACE_WFT = pickle.load(open('L1_replaces_WFT.p', 'rb'))
REPLACE_copexp = pickle.load(open('L1_replaces_copexp.p', 'rb'))
REPLACE_syn = pickle.load(open('L1_replaces_syn.p', 'rb'))
REPLACE_tensem = pickle.load(open('L1_replaces_tensem.p', 'rb'))
REPLACE_transliteration = pickle.load(open('L1_replaces_transliteration.p', 'rb'))

In [159]:
class Errorifier:
    """Generate errors in good sentences!"""

    def __init__(self, sentence: str):
        self.original_sentence = sentence.rstrip()
        self.sentence = self.original_sentence
        self.tokenized = None
        self.tokenize()

    def tokenize(self):
        self.tokenized = self.sentence.split()

    def correct(self):
        return self.original_sentence

    def no_error(self):
        return ' '.join(self.tokenized)

    def replace_error(self, redir=True, err_type='syn'):
        if err_type == 'WFT':
            COMMON_REPLACES = REPLACE_WFT
        if err_type == 'Copying expression':
            COMMON_REPLACES = REPLACE_copexp
        if err_type == 'Synonyms':
            COMMON_REPLACES = REPLACE_syn
        if err_type == 'Tense semantics':
            COMMON_REPLACES = REPLACE_tensem
        if err_type == 'Transliteration':
            COMMON_REPLACES = REPLACE_transliteration
        if len(self.tokenized) > 0:
            deletable = [i for i, w in enumerate(self.tokenized) if w in COMMON_REPLACES]
            index = random.choice(deletable)
            word = self.tokenized[index]
            if not COMMON_REPLACES[word]:
                return self.sentence

            # Normalize probabilities
            plist = list(COMMON_REPLACES[word].values())
            plistsum = sum(plist)
            plist = [x / plistsum for x in plist]

            # Choose a bad word
            repl = npchoice(list(COMMON_REPLACES[word].keys()), p=plist)
            self.tokenized[index] = '**' + repl + '**' + ' (' + err_type + ')'

        return ' '.join(self.tokenized)

    def error(self):
        """Introduce a random error."""
        count = npchoice([0,1,2,3,4],p=[0.05,0.07,0.25,0.35,0.28])
        for x in range(count):
            error_probs = [.20,.20,.20,.20,.20]
            error_type = npchoice(['WFT', 'Copying expression', 'Synonyms', 'Tense semantics', 'Transliteration'],p=error_probs)
            self.sentence = self.replace_error(err_type=error_type)
            self.tokenize()

        return self.sentence


In [ ]:
pipe = pipeline("text-generation", model=FINETUNED_GPT2_PATH) ## model after initial finetuning, not PPO-trained, path removed for anonymity

Testing cell

In [146]:
gen_test = pipe('Also,', num_return_sequences=10, return_full_text=True, do_sample=True, temperature=1)

In [154]:
with open('text_testing.txt', 'w') as f:
    for i in gen_test:
        f.write(i['generated_text'] + '\n')

Data corruption

In [160]:
!python3 error.py /content/PIE/errorify/text_testing.txt /content/

1it [00:00, 51.44it/s]
